# 🌍 GeoAI News Classifier
### Task 1 — NLP Pipeline + GIS Visualization

**Pipeline:**
1. Load AG News headlines (Hugging Face)
2. Classify topic with BERT (zero-shot via `transformers`)
3. Extract location mentions with spaCy NER
4. Geocode locations → lat/lon with GeoPy
5. Plot topic density on an interactive world map (Folium + Plotly)

---

## ⚙️ Cell 1 — Install Dependencies
Run once. Restart kernel after installation.

In [ ]:
# Install all required libraries
!pip install transformers datasets spacy geopy folium plotly torch -q
!python -m spacy download en_core_web_sm -q

print("✅ All dependencies installed. Restart kernel if running for the first time.")

## 📦 Cell 2 — Imports

In [ ]:
import time
import random
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# Hugging Face
from datasets import load_dataset
from transformers import pipeline

# spaCy NER
import spacy

# Geocoding
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Visualization
import folium
from folium.plugins import MarkerCluster, HeatMap
import plotly.express as px
import plotly.graph_objects as go

print("✅ All imports successful!")

## 📰 Cell 3 — Load AG News Dataset
AG News has **120,000 headlines** across 4 topics:
- `0` → World
- `1` → Sports  
- `2` → Business
- `3` → Sci/Tech

We sample **200 headlines** to keep geocoding fast.

In [ ]:
# Load AG News from Hugging Face (downloads ~30MB, cached after first run)
# Note: ag_news moved to fancyzhx/ag_news on newer versions of datasets library
print("Loading AG News dataset...")
try:
    dataset = load_dataset("fancyzhx/ag_news", split="test")  # new namespace
except Exception:
    dataset = load_dataset("ag_news", split="test")  # fallback for older versions

# Label mapping
LABEL_MAP = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# Topic colors for visualization
TOPIC_COLORS = {
    "World":    "#E74C3C",
    "Sports":   "#2ECC71",
    "Business": "#F39C12",
    "Sci/Tech": "#3498DB"
}

# Sample 200 headlines (50 per topic for balance)
random.seed(42)
samples = []
for label_id, label_name in LABEL_MAP.items():
    topic_items = [x for x in dataset if x['label'] == label_id]
    sampled = random.sample(topic_items, 50)
    for item in sampled:
        samples.append({
            'text': item['text'][:300],          # truncate long texts
            'true_label': label_name,
            'headline': item['text'].split('.')[0]  # first sentence as headline
        })

df = pd.DataFrame(samples)
print(f"✅ Loaded {len(df)} headlines")
print(f"\nTopic distribution:\n{df['true_label'].value_counts()}")
df.head()

## 🤖 Cell 4 — BERT Topic Classification (Zero-Shot)
Using `facebook/bart-large-mnli` for zero-shot classification — no fine-tuning needed.

> **Why zero-shot?** It's more impressive to demonstrate that BERT can classify *without* seeing labeled examples. We'll compare against the true AG News labels.

In [ ]:
print("Loading zero-shot classification pipeline (BART-large-MNLI)...")
print("⏳ First run downloads ~1.6GB model. Subsequent runs use cache.\n")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1  # CPU; change to 0 if you have GPU
)

CANDIDATE_LABELS = ["world news and politics", "sports and athletics", 
                    "business and finance", "science and technology"]

# Map model output back to our clean label names
LABEL_CLEAN = {
    "world news and politics": "World",
    "sports and athletics":    "Sports",
    "business and finance":    "Business",
    "science and technology":  "Sci/Tech"
}

print("Running BERT classification on 200 headlines...")
print("⏳ This takes ~3-8 minutes on CPU.\n")

predicted_labels = []
confidence_scores = []

for i, row in df.iterrows():
    result = classifier(row['text'], CANDIDATE_LABELS)
    top_label = LABEL_CLEAN[result['labels'][0]]
    top_score = result['scores'][0]
    predicted_labels.append(top_label)
    confidence_scores.append(round(top_score, 4))
    
    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/200 headlines...")

df['predicted_label'] = predicted_labels
df['confidence'] = confidence_scores

# Calculate accuracy
accuracy = (df['true_label'] == df['predicted_label']).mean()
print(f"\n✅ Classification complete!")
print(f"🎯 Zero-shot BERT Accuracy: {accuracy:.1%}")

df[['headline', 'true_label', 'predicted_label', 'confidence']].head(10)

## 📊 Cell 5 — Classification Results & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Classification report
print("=" * 55)
print("BERT Zero-Shot Classification Report")
print("=" * 55)
print(classification_report(df['true_label'], df['predicted_label']))

# Confusion matrix heatmap
labels_order = ["World", "Sports", "Business", "Sci/Tech"]
cm = confusion_matrix(df['true_label'], df['predicted_label'], labels=labels_order)

fig = px.imshow(
    cm,
    x=labels_order,
    y=labels_order,
    color_continuous_scale="Blues",
    title="🤖 BERT Zero-Shot Confusion Matrix",
    labels=dict(x="Predicted", y="True Label", color="Count"),
    text_auto=True
)
fig.update_layout(
    title_font_size=18,
    width=500, height=450
)
fig.show()

# Confidence distribution per topic
fig2 = px.box(
    df, x='predicted_label', y='confidence',
    color='predicted_label',
    color_discrete_map=TOPIC_COLORS,
    title="📊 Confidence Score Distribution by Topic",
    labels={'predicted_label': 'Topic', 'confidence': 'Confidence Score'}
)
fig2.update_layout(showlegend=False, width=600, height=400)
fig2.show()

## 🗺️ Cell 6 — spaCy NER: Extract Location Mentions
spaCy's `en_core_web_sm` model identifies:
- `GPE` → Countries, cities, states (most useful for us)
- `LOC` → Non-GPE locations (mountains, rivers, regions)
- `FAC` → Facilities, airports, highways

In [ ]:
print("Loading spaCy NER model...")
nlp = spacy.load("en_core_web_sm")

def extract_locations(text):
    """Extract GPE and LOC entities from text using spaCy NER."""
    doc = nlp(text)
    locations = []
    for ent in doc.ents:
        if ent.label_ in ("GPE", "LOC"):
            loc = ent.text.strip()
            # Filter out very short or numeric strings
            if len(loc) > 2 and not loc.isdigit():
                locations.append(loc)
    return list(set(locations))  # deduplicate per article

print("Extracting locations from all headlines...")
df['locations'] = df['text'].apply(extract_locations)
df['location_count'] = df['locations'].apply(len)

# Stats
articles_with_locations = (df['location_count'] > 0).sum()
total_mentions = df['location_count'].sum()

print(f"\n✅ NER complete!")
print(f"📍 Articles with at least 1 location: {articles_with_locations}/200")
print(f"📍 Total location mentions: {total_mentions}")
print(f"\n📌 Sample extractions:")

# Show examples
examples = df[df['location_count'] > 0][['headline', 'predicted_label', 'locations']].head(8)
for _, row in examples.iterrows():
    print(f"  [{row['predicted_label']}] {row['headline'][:70]}...")
    print(f"    → Locations: {row['locations']}\n")

## 🌐 Cell 7 — Geocode Locations with GeoPy
Using **Nominatim** (OpenStreetMap) — free, no API key needed.

> ⚠️ **Rate limit:** 1 request/second. We cache results to avoid redundant calls.

In [ ]:
geolocator = Nominatim(user_agent="geoai_news_classifier_internship_v1")

# Cache to avoid re-geocoding the same place
geocache = {}

def geocode_location(location_name, retries=2):
    """Geocode a location string → (lat, lon) or None."""
    if location_name in geocache:
        return geocache[location_name]
    
    for attempt in range(retries):
        try:
            time.sleep(1.1)  # Respect Nominatim rate limit
            location = geolocator.geocode(location_name, timeout=10)
            if location:
                result = (location.latitude, location.longitude)
                geocache[location_name] = result
                return result
        except (GeocoderTimedOut, GeocoderServiceError):
            time.sleep(2)
    
    geocache[location_name] = None
    return None

# Collect all unique locations across the dataset
all_unique_locations = set()
for locs in df['locations']:
    all_unique_locations.update(locs)

print(f"Geocoding {len(all_unique_locations)} unique locations...")
print("⏳ ~1 second per location due to rate limiting.\n")

for i, loc in enumerate(sorted(all_unique_locations)):
    result = geocode_location(loc)
    status = f"✓ ({result[0]:.2f}, {result[1]:.2f})" if result else "✗ not found"
    if (i+1) % 10 == 0:
        print(f"  [{i+1}/{len(all_unique_locations)}] {loc}: {status}")
    else:
        print(f"  {loc}: {status}")

found = sum(1 for v in geocache.values() if v is not None)
print(f"\n✅ Geocoding complete! {found}/{len(all_unique_locations)} locations resolved.")

## 🗃️ Cell 8 — Build GeoDataFrame for Plotting
Explode the data so each (article × location) pair is one row.

In [ ]:
# Build rows: one per (article, location) pair
geo_rows = []

for _, row in df.iterrows():
    for loc in row['locations']:
        coords = geocache.get(loc)
        if coords:  # only include successfully geocoded locations
            geo_rows.append({
                'headline':  row['headline'][:80],
                'topic':     row['predicted_label'],
                'confidence': row['confidence'],
                'location':  loc,
                'lat':       coords[0],
                'lon':       coords[1],
                'color':     TOPIC_COLORS[row['predicted_label']]
            })

geo_df = pd.DataFrame(geo_rows)

print(f"✅ GeoDataFrame built: {len(geo_df)} plotable (article, location) pairs")
print(f"\nTopic breakdown of geocoded mentions:")
print(geo_df['topic'].value_counts())
print()
geo_df.head(10)

## 🗺️ Cell 9 — Interactive Map: Topic-Colored Markers (Folium)
Each marker = a location mentioned in a news article, colored by topic.

In [ ]:
# Folium topic color mapping (uses named colors)
FOLIUM_COLORS = {
    "World":    "red",
    "Sports":   "green",
    "Business": "orange",
    "Sci/Tech": "blue"
}

# Base map
m = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="CartoDB positron"  # clean light basemap
)

# Add MarkerCluster per topic for clean rendering
topic_clusters = {}
for topic in LABEL_MAP.values():
    topic_clusters[topic] = MarkerCluster(name=f"{topic} News").add_to(m)

# Add markers
for _, row in geo_df.iterrows():
    popup_html = f"""
    <div style='font-family: Arial; width: 220px;'>
        <b style='color: {row['color']}'>[{row['topic']}]</b><br>
        <i>{row['headline']}...</i><br><br>
        <b>📍 Location:</b> {row['location']}<br>
        <b>🎯 Confidence:</b> {row['confidence']:.1%}
    </div>
    """
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7,
        color=row['color'],
        fill=True,
        fill_color=row['color'],
        fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"[{row['topic']}] {row['location']}"
    ).add_to(topic_clusters[row['topic']])

# Legend
legend_html = """
<div style="position: fixed; bottom: 40px; left: 40px; z-index: 9999;
            background: white; padding: 14px 18px; border-radius: 8px;
            border: 2px solid #ccc; font-family: Arial; font-size: 13px;
            box-shadow: 2px 2px 6px rgba(0,0,0,0.2);">
  <b>📰 News Topic</b><br><br>
  <span style='color:#E74C3C'>●</span> World &nbsp;&nbsp;
  <span style='color:#2ECC71'>●</span> Sports<br><br>
  <span style='color:#F39C12'>●</span> Business &nbsp;
  <span style='color:#3498DB'>●</span> Sci/Tech
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# Save
m.save("news_topic_map.html")
print("✅ Interactive map saved to: news_topic_map.html")
print("   Open in browser to explore!")
m  # renders inline in Jupyter

## 🔥 Cell 10 — Heatmap: News Density by Location

In [ ]:
# Heatmap — shows overall geographic density of news coverage
heat_map = folium.Map(
    location=[20, 0],
    zoom_start=2,
    tiles="CartoDB dark_matter"  # dark basemap looks great for heatmaps
)

# Heat data: [lat, lon, weight] — weight = confidence score
heat_data = [[row['lat'], row['lon'], row['confidence']] 
             for _, row in geo_df.iterrows()]

HeatMap(
    heat_data,
    min_opacity=0.4,
    max_zoom=5,
    radius=25,
    blur=18,
    gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(heat_map)

# Title
title_html = """
<div style="position: fixed; top: 15px; left: 50%; transform: translateX(-50%);
            z-index: 9999; background: rgba(0,0,0,0.7); color: white;
            padding: 10px 20px; border-radius: 6px; font-family: Arial;
            font-size: 15px; font-weight: bold;">
  🌍 Global News Mention Density (BERT + spaCy + GeoPy)
</div>
"""
heat_map.get_root().html.add_child(folium.Element(title_html))

heat_map.save("news_heatmap.html")
print("✅ Heatmap saved to: news_heatmap.html")
heat_map

## 📊 Cell 11 — Plotly: Topic Density on World Map (Bubble Map)

In [ ]:
# Aggregate: count mentions per location per topic
location_counts = (
    geo_df.groupby(['location', 'topic', 'lat', 'lon'])
    .size()
    .reset_index(name='mention_count')
)

fig = px.scatter_geo(
    location_counts,
    lat='lat',
    lon='lon',
    color='topic',
    size='mention_count',
    hover_name='location',
    hover_data={'lat': False, 'lon': False, 'mention_count': True, 'topic': True},
    color_discrete_map=TOPIC_COLORS,
    size_max=40,
    projection='natural earth',
    title='🌍 News Topic Density — BERT + spaCy + GeoPy Pipeline',
)

fig.update_layout(
    title_font_size=17,
    legend_title_text='Topic',
    geo=dict(
        showland=True,
        landcolor='#F0F0E8',
        showocean=True,
        oceancolor='#D0E8F0',
        showcoastlines=True,
        coastlinecolor='#AAAAAA',
        showframe=False,
        showcountries=True,
        countrycolor='#CCCCCC',
        bgcolor='#FAFAFA'
    ),
    height=600,
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()
fig.write_html("news_bubble_map.html")
print("✅ Bubble map saved to: news_bubble_map.html")

## 📊 Cell 12 — Bar Chart: Top Mentioned Countries per Topic

In [ ]:
# Top 10 most mentioned locations overall
top_locations = (
    geo_df.groupby(['location', 'topic'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

fig = px.bar(
    top_locations,
    x='count',
    y='location',
    color='topic',
    orientation='h',
    color_discrete_map=TOPIC_COLORS,
    title='📍 Top 20 Most Mentioned Locations (by News Topic)',
    labels={'count': 'Mention Count', 'location': 'Location', 'topic': 'Topic'}
)

fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=550,
    legend_title_text='Topic',
    title_font_size=16
)
fig.show()

## 📊 Cell 13 — Choropleth: Topic Dominance per Country

In [ ]:
# Use pycountry to map location names to ISO-3 codes for choropleth
try:
    import pycountry
except ImportError:
    !pip install pycountry -q
    import pycountry

def get_country_code(name):
    """Try to get ISO-3 country code from a name string."""
    try:
        results = pycountry.countries.search_fuzzy(name)
        return results[0].alpha_3
    except:
        return None

# Get dominant topic per location
dominant_topic = (
    geo_df.groupby(['location', 'topic'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .groupby('location')
    .first()
    .reset_index()
)

print("Mapping locations to country ISO codes...")
dominant_topic['iso_code'] = dominant_topic['location'].apply(get_country_code)
choropleth_df = dominant_topic.dropna(subset=['iso_code'])

# Encode topic as number for color scale
topic_num = {"World": 1, "Sports": 2, "Business": 3, "Sci/Tech": 4}
choropleth_df = choropleth_df.copy()
choropleth_df['topic_num'] = choropleth_df['topic'].map(topic_num)

fig = px.choropleth(
    choropleth_df,
    locations='iso_code',
    color='topic',
    hover_name='location',
    hover_data={'iso_code': False, 'count': True, 'topic_num': False},
    color_discrete_map=TOPIC_COLORS,
    title='🗺️ Dominant News Topic per Country/Region',
    projection='natural earth'
)

fig.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    height=500,
    title_font_size=16,
    margin=dict(l=0, r=0, t=50, b=0)
)
fig.show()
print(f"✅ Choropleth rendered for {len(choropleth_df)} countries/regions.")

## 🧾 Cell 14 — Final Summary & Pipeline Recap

In [ ]:
print("=" * 60)
print("        GeoAI NEWS CLASSIFIER — PIPELINE SUMMARY")
print("=" * 60)

print(f"""
📰 DATASET
   Source      : AG News (Hugging Face)
   Headlines   : {len(df)}
   Topics      : World, Sports, Business, Sci/Tech

🤖 BERT CLASSIFICATION (Zero-Shot)
   Model       : facebook/bart-large-mnli
   Accuracy    : {(df['true_label'] == df['predicted_label']).mean():.1%}
   Avg. Conf.  : {df['confidence'].mean():.1%}

📍 SPACY NER (Location Extraction)
   Model       : en_core_web_sm
   Entity types: GPE, LOC
   Articles with locations : {(df['location_count'] > 0).sum()}/{len(df)}
   Total location mentions : {df['location_count'].sum()}

🌐 GEOPY GEOCODING
   Service     : Nominatim (OpenStreetMap, free)
   Unique locs : {len(all_unique_locations)}
   Resolved    : {sum(1 for v in geocache.values() if v)}
   Plotable pts: {len(geo_df)}

🗺️ VISUALIZATIONS GENERATED
   1. news_topic_map.html  → Interactive marker map (Folium)
   2. news_heatmap.html    → Density heatmap (Folium)
   3. news_bubble_map.html → Bubble map by topic (Plotly)
   4. Bar chart            → Top locations per topic (Plotly)
   5. Choropleth           → Dominant topic per country (Plotly)
""")
print("=" * 60)
print("✅ Pipeline complete! All outputs saved.")